# Code mẫu — Random forest

Random forest dựa trên hai ý: mỗi cây học trên một mẫu bootstrap, và các cây được gộp lại. Ví dụ đầu làm một lượt bootstrap bằng tay. Ví dụ sau đo xem độ đa dạng giữa các cây ảnh hưởng tới điểm validation ra sao. Ví dụ cuối đọc feature importance, và nhắc rằng nó chỉ mô tả model, không phải kết luận về nguyên nhân.

## Ví dụ 1: Một lượt bootstrap và hai cách gộp

Có năm mẫu, đánh số từ 0 đến 4. Một cây rút năm lần, có hoàn lại. Mẫu nào không được rút lần nào là mẫu OOB của cây đó, tức mẫu cây chưa từng thấy. Sau đó ta lấy ba dự đoán xác suất của ba cây, và so hai cách gộp: lấy trung bình xác suất, và bỏ phiếu theo nhãn.

In [ ]:
import numpy as np

rng = np.random.default_rng(7)
draw = rng.integers(0, 5, size=5)
seen = np.unique(draw)
oob = np.setdiff1d(np.arange(5), seen)
print("rút:", draw.tolist())
print("đã thấy:", seen.tolist(), "OOB:", oob.tolist())

tree_probabilities = np.array([0.9, 0.4, 0.4])
mean_probability = tree_probabilities.mean()
votes_for_one = int(np.sum(tree_probabilities > 0.5))
print("trung bình xác suất:", round(float(mean_probability), 3))
print("phiếu lớp 1:", votes_for_one, "/", len(tree_probabilities))
assert np.all(np.isin(oob, [0, 1, 2, 3, 4]))
assert np.isclose(mean_probability, 17 / 30)
assert votes_for_one == 1

rút: [4, 3, 3, 4, 2]
đã thấy: [2, 3, 4] OOB: [0, 1]
trung bình xác suất: 0.567
phiếu lớp 1: 1 / 3


Rút đủ năm lần nhưng cây chỉ thấy ba mẫu, vì mẫu 3 và mẫu 4 đều bị rút hai lần. Mẫu 0 và mẫu 1 thành OOB.

Hai cách gộp cho hai kết quả ngược nhau. Trung bình xác suất là 0.567, vượt ngưỡng 0.5, nên chọn lớp 1. Bỏ phiếu thì chỉ một trong ba cây nghiêng về lớp 1, nên chọn lớp 0. Cây đầu rất chắc chắn, còn hai cây sau chỉ hơi nghiêng về lớp 0. Trung bình xác suất giữ lại thông tin về mức chắc chắn ấy, còn bỏ phiếu thì làm mất. Random forest trong scikit-learn dùng cách lấy trung bình xác suất.

Vì sao các cây cần khác nhau? Một công thức đơn giản trả lời được câu này. Giả sử mỗi cây có phương sai bằng 1, mọi cặp cây có cùng hệ số tương quan `rho`, và rừng có 100 cây. Khi đó phương sai của trung bình là `rho + (1 − rho) / 100`.

In [ ]:
for rho in [0.5, 0.1]:
    variance_of_mean = rho + (1. - rho) / 100
    print("rho =", rho, "phương sai trung bình:", round(variance_of_mean, 3))
assert np.isclose(0.5 + 0.5 / 100, 0.505)
assert np.isclose(0.1 + 0.9 / 100, 0.109)

rho = 0.5 phương sai trung bình: 0.505
rho = 0.1 phương sai trung bình: 0.109


Thêm cây chỉ làm nhỏ phần `(1 − rho) / 100`. Phần `rho` thì đứng yên, dù có bao nhiêu cây. Các cây giống nhau tới mức tương quan 0.5 thì phương sai không xuống dưới 0.5 được. Cây khác nhau hơn, tương quan 0.1, thì phương sai xuống còn 0.109. Công thức này chỉ đúng dưới giả thiết mọi cây cùng phương sai và cùng tương quan. Nó giải thích ý tưởng, chứ không hứa điểm số cho một rừng cụ thể.

## Ví dụ 2: Đo OOB và độ đa dạng trên tập giữ lại

Dữ liệu được chia thành train và validation ngay từ đầu. Hai rừng có cùng 60 cây, chỉ khác `max_features`. Rừng thứ nhất cho mỗi nút xét mọi cột. Rừng thứ hai chỉ cho xét ngẫu nhiên khoảng `sqrt(d)` cột. Để đo độ đa dạng, ta lấy ba cặp cây và đếm tỉ lệ mẫu validation mà hai cây đoán khác nhau.

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

X, y = make_classification(
    n_samples=220, n_features=6, n_informative=3, n_redundant=0,
    class_sep=0.8, flip_y=0.08, random_state=12
)
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=4
)

for label, max_features in [("all", 1.0), ("sqrt", "sqrt")]:
    forest = RandomForestClassifier(
        n_estimators=60, max_features=max_features, oob_score=True,
        random_state=0, n_jobs=1
    ).fit(X_train, y_train)
    tree_predictions = np.array([
        tree.predict(X_valid) for tree in forest.estimators_
    ])
    pair_disagreements = [
        np.mean(tree_predictions[i] != tree_predictions[j])
        for i, j in [(0, 1), (0, 2), (1, 2)]
    ]
    print(label, "valid:", round(accuracy_score(y_valid, forest.predict(X_valid)), 3),
          "OOB:", round(float(forest.oob_score_), 3),
          "bất đồng 3 cặp:", np.round(pair_disagreements, 3).tolist())
    assert len(forest.estimators_) == 60

all valid: 0.782 OOB: 0.727 bất đồng 3 cặp: [0.2, 0.255, 0.164]
sqrt valid: 0.8 OOB: 0.739 bất đồng 3 cặp: [0.309, 0.291, 0.273]


Khi chỉ được xét `sqrt(d)` cột ở mỗi nút, các cặp cây bất đồng nhiều hơn, khoảng 0.27 đến 0.31 so với 0.16 đến 0.26. Lần này rừng đa dạng hơn cũng có điểm cao hơn, cả trên validation lẫn OOB. Nhưng không phải lúc nào cũng vậy. Mỗi cây bị giới hạn cột thì thường yếu hơn, nên độ bất đồng phải đọc cùng điểm validation, đừng cố đẩy nó lên một mình.

OOB score là điểm được tính miễn phí trong lúc train. Mỗi hàng train chỉ được chấm bởi những cây không rút trúng nó. Vì vậy OOB đóng vai trò như một validation có sẵn, dù ở đây ta vẫn giữ một tập validation riêng để so.

## Ví dụ 3: Importance, cột tương quan và một thay đổi có kiểm soát

Hai cột đầu cùng mang một tín hiệu. `tin_hieu_a` là tín hiệu gốc, còn `tin_hieu_b` là bản sao có thêm chút nhiễu. Cột `nhieu` thì không liên quan gì tới nhãn. Ta so impurity importance với permutation importance trên validation. Sau đó giữ nguyên cách chia và thử ba giá trị `min_samples_leaf`.

In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(23)
n = 240
tin_hieu_a = rng.normal(size=n)
tin_hieu_b = tin_hieu_a + rng.normal(0., 0.35, n)
nhieu = rng.normal(size=n)
y = (tin_hieu_a + rng.normal(0., 0.8, n) > 0).astype(int)
X = np.column_stack([tin_hieu_a, tin_hieu_b, nhieu])
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=5
)

forest = RandomForestClassifier(
    n_estimators=80, max_features="sqrt", random_state=0, n_jobs=1
).fit(X_train, y_train)
permutation = permutation_importance(
    forest, X_valid, y_valid, n_repeats=10, random_state=0,
    scoring="accuracy", n_jobs=1
)
print("valid:", round(accuracy_score(y_valid, forest.predict(X_valid)), 3))
print("impurity:", forest.feature_importances_.round(3).tolist())
print("permutation valid:", permutation.importances_mean.round(3).tolist())
assert permutation.importances_mean.shape == (3,)

for leaf_size in [1, 5, 12]:
    candidate = RandomForestClassifier(
        n_estimators=80, max_features="sqrt", min_samples_leaf=leaf_size,
        random_state=0, n_jobs=1
    ).fit(X_train, y_train)
    print("min_samples_leaf =", leaf_size,
          "train:", round(accuracy_score(y_train, candidate.predict(X_train)), 3),
          "valid:", round(accuracy_score(y_valid, candidate.predict(X_valid)), 3))

valid: 0.806
impurity: [0.371, 0.408, 0.221]
permutation valid: [0.171, 0.144, 0.031]
min_samples_leaf = 1 train: 1.0 valid: 0.806
min_samples_leaf = 5 train: 0.875 valid: 0.806
min_samples_leaf = 12 train: 0.815 valid: 0.792


Có hai điều đáng để ý ở dòng impurity. Thứ nhất, `tin_hieu_b` đứng trên `tin_hieu_a`, dù nhãn được tạo từ `tin_hieu_a`. Hai cột gần như giống nhau, nên cây nào dùng cột nào phần lớn là tình cờ. Thứ hai, cột nhiễu vẫn nhận 0.221. Impurity importance tính trên train, và cây sâu luôn tìm được vài ngưỡng có vẻ hữu ích trên một cột nhiễu.

Permutation importance trên validation hạ cột nhiễu xuống còn 0.031. Còn với hai cột tín hiệu, xáo trộn riêng một cột chỉ làm điểm giảm vừa phải, vì cột kia vẫn còn giữ tín hiệu. Điểm giảm ít ở đây không có nghĩa cột đó vô ích. Cả hai cách đo chỉ nói rừng đã dựa vào cột nào trên dữ liệu này, không nói cột nào gây ra nhãn.

Ba dòng cuối là một thí nghiệm có kiểm soát: cùng dữ liệu, cùng seed, cùng cách chia, chỉ thay độ mịn của lá. Lá to hơn làm điểm train giảm rõ, còn validation giữ nguyên ở 0.806 rồi mới giảm khi lá lên 12. Ta chọn cấu hình theo validation. Muốn có một con số cuối cùng để báo cáo, cần thêm một tập test tách riêng.